# HW1 Part 2 — Earnings-Call Signal Evaluation

Consumes the call-level JSONs produced by `notebook/extraction.ipynb` (Stage D) and turns them into features, labels, predictions, and a backtest on +1d / +5d / +21d / +63d forward excess returns.

## Phase gate

Set the `PHASE` variable in the cell below to control how much of the pipeline runs:

| Phase | Scope                                                                                       | Runtime |
|-------|---------------------------------------------------------------------------------------------|---------|
| `E0`  | 4 tickers × 2 quarters, ext_finbert_llm only, +5d only, forced τ=0.0 — orchestration smoke | ~30 s   |
| `E1`  | Full 14-ticker × 131-call corpus, ext_finbert_llm, +5d and +21d                              | ~1 min  |
| `E2`  | E1 + ext_lm_llm (requires `data/external/LM_dict.csv`)                                       | ~1 min  |
| `E3`  | E2 + ext_finbert_only ablation                                                               | ~1 min  |

Every cell is orchestration only — the heavy lifting lives in `src/*.py` (Plan rule: "notebooks orchestrate only"). Cache freshness is driven by `src/cache_keys.py` AST fingerprints, so re-running after an edit to any `src/*.py` file automatically prunes the stale artifacts.

In [1]:
PHASE = "E3"  # E0 / E1 / E2 / E3

import sys, warnings
from pathlib import Path
warnings.filterwarnings("ignore")
sys.path.insert(0, str(Path.cwd().parent))

from src import (
    aggregate,
    analysis,
    backtest,
    baselines,
    cache_keys,
    dataset,
    features,
    io_paths,
    labels,
    lexicon_lm,
    models,
    parser,
    plots,
    prices,
    qc,
)

io_paths.ensure_dirs()

E0_SUBSET = [
    ("AMD",  "Q3-2024"),
    ("AMD",  "Q4-2024"),
    ("PLTR", "Q4-2024"),
    ("PLTR", "Q1-2025"),
    ("NVDA", "Q4-2024"),
    ("NVDA", "Q1-2025"),
    ("JNJ",  "Q1-2024"),
    ("JNJ",  "Q2-2024"),
]

if PHASE == "E0":
    CALLS = E0_SUBSET
    EXTRACTIONS = ("ext_finbert_llm",)
    HORIZONS = ("5d",)
    INCLUDE_LM = False
    FORCE_E0_TAU = True
    PRIMARY_K = 1
elif PHASE == "E1":
    CALLS = [io_paths.parse_stem(p) for p in io_paths.list_transcripts()]
    EXTRACTIONS = ("ext_finbert_llm",)
    HORIZONS = ("5d", "21d")
    INCLUDE_LM = False
    FORCE_E0_TAU = False
    PRIMARY_K = 5
elif PHASE == "E2":
    CALLS = [io_paths.parse_stem(p) for p in io_paths.list_transcripts()]
    EXTRACTIONS = ("ext_finbert_llm", "ext_lm_llm")
    HORIZONS = ("5d", "21d")
    INCLUDE_LM = True
    FORCE_E0_TAU = False
    PRIMARY_K = 5
elif PHASE == "E3":
    CALLS = [io_paths.parse_stem(p) for p in io_paths.list_transcripts()]
    EXTRACTIONS = ("ext_finbert_llm", "ext_lm_llm", "ext_finbert_only")
    HORIZONS = ("5d", "21d")
    INCLUDE_LM = True
    FORCE_E0_TAU = False
    PRIMARY_K = 5
else:
    io_paths.error(f"unknown PHASE: {PHASE}")

print(f"PHASE={PHASE}  n_calls={len(CALLS)}  extractions={EXTRACTIONS}  horizons={HORIZONS}  include_lm={INCLUDE_LM}")

PHASE=E3  n_calls=131  extractions=('ext_finbert_llm', 'ext_lm_llm', 'ext_finbert_only')  horizons=('5d', '21d')  include_lm=True


## Stage P — Prices

Pull daily Close for all 14 corpus tickers plus the SPY benchmark from yfinance. Cached as parquet under `data/cache/prices/`; subsequent runs are instant no-ops. Set `refresh=True` to force a re-download (not recommended — yfinance nightly-patches history and would shift labels).

In [2]:
_ = prices.fetch_corpus_prices()

  prices[  AMD] rows= 658  2023-09-01 → 2026-04-17
  prices[ AVGO] rows= 658  2023-09-01 → 2026-04-17
  prices[  BLK] rows= 658  2023-09-01 → 2026-04-17
  prices[    C] rows= 658  2023-09-01 → 2026-04-17
  prices[ FAST] rows= 658  2023-09-01 → 2026-04-17
  prices[  FDX] rows= 658  2023-09-01 → 2026-04-17
  prices[   GS] rows= 658  2023-09-01 → 2026-04-17
  prices[ INTC] rows= 658  2023-09-01 → 2026-04-17
  prices[  JNJ] rows= 658  2023-09-01 → 2026-04-17
  prices[  JPM] rows= 658  2023-09-01 → 2026-04-17
  prices[  NKE] rows= 658  2023-09-01 → 2026-04-17
  prices[ NVDA] rows= 658  2023-09-01 → 2026-04-17
  prices[ PLTR] rows= 658  2023-09-01 → 2026-04-17
  prices[  WFC] rows= 658  2023-09-01 → 2026-04-17
  prices[  SPY] rows= 658  2023-09-01 → 2026-04-17


## Stage B-LM — LM sentiment (only for E2/E3)

Score every unit against the Loughran-McDonald master dictionary. Cached at `data/cache/sentiment_lm/<unit_id>.json` with a stage fingerprint, so an edit to `src/lexicon_lm.py` wipes all stale files automatically.

In [3]:
if INCLUDE_LM:
    pruned = cache_keys.prune_stale("sentiment_lm")
    if pruned:
        print(f"pruned {pruned} stale sentiment_lm records")
    all_units = []
    for ticker, quarter in CALLS:
        all_units.extend(parser.read_units(ticker, quarter))
    scored = lexicon_lm.score_units_lm(all_units)
    n_with_signal = sum(1 for r in scored if r.get("sentiment") is not None)
    print(f"LM scored n={len(scored)} units  with_signal={n_with_signal}  null={len(scored) - n_with_signal}")
else:
    print("skip (PHASE<E2)")

LM scored n=3339 units  with_signal=2750  null=589


## Stage D-v2 — Call-level aggregation with LM variant

FinBERT call JSONs are assumed fresh (produced by `extraction.ipynb` Stage D). If they're missing `_cache_key` or the fingerprint no longer matches, the next two lines rebuild them. The LM variant builds a parallel `calls_lm/<ticker>_<quarter>.json` set when `INCLUDE_LM=True`.

In [4]:
import time
t0 = time.time()
pruned_fb = cache_keys.prune_stale("calls")
n_fb_built = 0
for ticker, quarter in CALLS:
    try:
        aggregate.load_call_record(ticker, quarter, variant="finbert")
    except RuntimeError:
        aggregate.build_and_write(ticker, quarter, variant="finbert")
        n_fb_built += 1
print(f"finbert calls: pruned={pruned_fb}  rebuilt={n_fb_built}  elapsed={time.time()-t0:.1f}s")

if INCLUDE_LM:
    t0 = time.time()
    pruned_lm = cache_keys.prune_stale("calls_lm")
    n_lm_built = 0
    for ticker, quarter in CALLS:
        try:
            aggregate.load_call_record(ticker, quarter, variant="lm")
        except RuntimeError:
            aggregate.build_and_write(ticker, quarter, variant="lm")
            n_lm_built += 1
    print(f"lm calls:      pruned={pruned_lm}  rebuilt={n_lm_built}  elapsed={time.time()-t0:.1f}s")

finbert calls: pruned=0  rebuilt=0  elapsed=0.7s


lm calls:      pruned=0  rebuilt=0  elapsed=0.9s


## Stage F — Features + Labels

Feature table is one row per `(ticker, call_date)` with 31 columns (FinBERT only) or 39 columns (FinBERT + LM). Labels are +1d / +5d / +21d / +63d forward raw + excess returns. Both parquets carry source-fingerprint sidecars for cache invalidation.

In [5]:
cache_keys.prune_stale("features")
cache_keys.prune_stale("labels")
feat_df = features.build_feature_table(CALLS, include_lm=INCLUDE_LM)
labels_df = labels.compute_labels(CALLS)
print(f"features: rows={len(feat_df)} cols={len(feat_df.columns)}")
print(f"labels:   rows={len(labels_df)}")

features: rows=131 cols=39
labels:   rows=131


## Stage QC — Part 2 quality gates

Four checks:

1. `sentiment_lm_nan_rate` — fraction of text-bearing units with zero LM dict hits (target ≤ 25%; LM is sparse).
2. `label_nan_rate_by_horizon` — per-horizon NaN fraction (tightening at longer horizons is expected).
3. `ret_21d_prior_nan_count` — boundary feature NaN count (target ≤ 2 — only the very first call of each ticker can fall below the 22-day prices window).
4. `run_cells_nan_rate` — all model output parquets must have 0% NaN scores; classifier probabilities must be in `[0, 1]`.

In [6]:
qc_report = {}

if INCLUDE_LM:
    all_units_qc = []
    for ticker, quarter in CALLS:
        all_units_qc.extend(parser.read_units(ticker, quarter))
    qc_report["sentiment_lm"] = qc.sentiment_lm_quality(all_units_qc)

qc_report["labels"] = qc.label_quality(labels_df)
qc_report["features_momentum"] = qc.feature_momentum_quality(feat_df)

for section, payload in qc_report.items():
    print(f"[{section}]")
    for k, v in payload.items():
        print(f"  {k}: {v}")
    print()

_ = qc.save_qc_report(f"part2_{PHASE}", qc_report)

[sentiment_lm]
  n_seen: 3324
  n_nan: 574
  nan_rate: 0.1726835138387485

[labels]
  n_rows: 131
  horizons: ['1d', '5d', '21d', '63d']
  nan_rate_by_horizon: {'1d': 0.0, '5d': 0.03816793893129771, '21d': 0.05343511450381679, '63d': 0.1297709923664122}

[features_momentum]
  ret_21d_prior_nan_count: 0
  n_rows: 131



## Stage M — Train + backtest across every (extraction, horizon, model) cell

For each extraction variant × horizon combo:

1. Split primary (per-ticker first 5 calls → train, rest → test).
2. Canonicalize the frame so every variant exposes `sentiment_call` as the unifying sentiment column.
3. Fit rule (τ from `pick_tau`), logreg, xgb, ridge; persist predictions, equity curve, and metrics.
4. Append/replace the row in `data/cache/backtest/summary.parquet`.

E0 forces τ=0.0

In [7]:
import pandas as pd

joined = dataset.join_features_labels(feat_df, labels_df)
train_raw, test_raw = dataset.split_primary(joined, k=PRIMARY_K)
print(f"primary split (k={PRIMARY_K}): train={len(train_raw)}  test={len(test_raw)}")

run_cells_qc: list[dict] = []

for extraction in EXTRACTIONS:
    train_df = dataset.canonicalize_variant(train_raw, extraction)
    test_df = dataset.canonicalize_variant(test_raw, extraction)

    for horizon in HORIZONS:
        if FORCE_E0_TAU:
            rule_params = baselines.e0_rule_params(extraction)
        else:
            rule_params = baselines.pick_tau(train_df, extraction, horizon)

        test_nonan = test_df[test_df[f"excess_{horizon}"].notna()].reset_index(drop=True)
        if test_nonan.empty:
            print(f"{extraction} {horizon}: no test rows, skipping")
            continue
        action_te, score_te = baselines.rule_predict(test_nonan, rule_params)
        _, _, meta_te_rule = dataset.build_xy(test_df, horizon, target="binary")
        backtest.run_backtest(
            meta_te_rule, score_te, action_te,
            horizon=horizon, extraction=extraction, model_name="rule", score_kind="rule",
        )

        Xtr, ytr, _ = dataset.build_xy(train_df, horizon, target="binary")
        Xte, yte, meta_te = dataset.build_xy(test_df, horizon, target="binary")

        logreg_m = models.fit_logreg(Xtr, ytr)
        logreg_scores = pd.Series(models.predict(logreg_m, Xte, "binary"), index=Xte.index, name="score")
        models.save_model(logreg_m, extraction, "logreg", horizon)
        backtest.run_backtest(meta_te, logreg_scores, None, horizon=horizon,
                              extraction=extraction, model_name="logreg", score_kind="binary")

        xgb_m = models.fit_xgb(Xtr, ytr)
        xgb_scores = pd.Series(models.predict(xgb_m, Xte, "binary"), index=Xte.index, name="score")
        models.save_model(xgb_m, extraction, "xgb", horizon)
        backtest.run_backtest(meta_te, xgb_scores, None, horizon=horizon,
                              extraction=extraction, model_name="xgb", score_kind="binary")

        Xtr_r, ytr_r, _ = dataset.build_xy(train_df, horizon, target="regression")
        Xte_r, yte_r, meta_te_r = dataset.build_xy(test_df, horizon, target="regression")
        ridge_m = models.fit_ridge(Xtr_r, ytr_r)
        ridge_scores = pd.Series(models.predict(ridge_m, Xte_r, "regression"), index=Xte_r.index, name="score")
        models.save_model(ridge_m, extraction, "ridge", horizon)
        backtest.run_backtest(meta_te_r, ridge_scores, None, horizon=horizon,
                              extraction=extraction, model_name="ridge", score_kind="regression")

        for model_name, score_kind in (("rule", "rule"), ("logreg", "binary"), ("xgb", "binary"), ("ridge", "regression")):
            preds_df = pd.read_parquet(io_paths.preds_path(extraction, model_name, horizon))
            cell_qc = qc.run_cells_quality(preds_df, score_kind=score_kind)
            run_cells_qc.append({"extraction": extraction, "horizon": horizon, "model": model_name, **cell_qc})

print(f"\ntrained {len(run_cells_qc)} (extraction, horizon, model) cells")

primary split (k=5): train=70  test=61



trained 24 (extraction, horizon, model) cells


## Stage R — Summary + run-cells QC

Print `summary.parquet` filtered to the runs produced by *this* phase, plus the per-cell run QC. Anything in the "path" column of run-cells with `nan_rate > 0.1`, `std_zero=True`, or `prob_out_of_range=True` is a red flag.

In [8]:
summary = pd.read_parquet(io_paths.backtest_summary_path())
mask = summary["extraction"].isin(EXTRACTIONS) & summary["horizon"].isin(HORIZONS)
phase_summary = summary[mask].copy()
show_cols = [
    "extraction", "model", "horizon",
    "n_trades", "directional_accuracy", "hit_rate",
    "IC_spearman", "pnl_sum", "sharpe_annualized", "final_equity",
]
phase_summary = phase_summary.sort_values(["horizon", "extraction", "model"]).reset_index(drop=True)
print("=== Backtest summary (phase-filtered) ===")
print(phase_summary[show_cols].to_string(index=False))

print("\n=== Run-cells QC ===")
rc_df = pd.DataFrame(run_cells_qc)
if rc_df.empty:
    print("(no run-cells produced — check that test set is non-empty)")
else:
    print(rc_df.to_string(index=False))
    bad = rc_df[(rc_df.nan_rate > 0.1) | rc_df.std_zero | (rc_df.prob_out_of_range == True)]
    if len(bad):
        print("\n!!! run-cells with pathologies:")
        print(bad.to_string(index=False))
    else:
        print("\nrun-cells: all clean")

=== Backtest summary (phase-filtered) ===
      extraction  model horizon  n_trades  directional_accuracy  hit_rate  IC_spearman   pnl_sum  sharpe_annualized  final_equity
 ext_finbert_llm logreg     21d        54              0.500000  0.481481    -0.038384  0.018693           0.034206      0.878376
 ext_finbert_llm  ridge     21d        54              0.481481  0.481481    -0.128416  0.018693           0.034206      0.878376
 ext_finbert_llm   rule     21d        21              0.444444  0.476190     0.093044  0.010301           0.029569      0.951801
 ext_finbert_llm    xgb     21d        54              0.500000  0.481481    -0.083896  0.018693           0.034206      0.878376
ext_finbert_only logreg     21d        54              0.444444  0.481481    -0.120335  0.018693           0.034206      0.878376
ext_finbert_only  ridge     21d        54              0.481481  0.481481    -0.152201  0.018693           0.034206      0.878376
ext_finbert_only   rule     21d        54       

## Stage R2 — Per-ticker hit rate (Part II deliverable §3)

For the main `(ext_finbert_llm, +5d)` cell: per-ticker number of long trades and hit rate. Part II §3 asks this table in the report.

In [9]:
if "ext_finbert_llm" in EXTRACTIONS and "5d" in HORIZONS:
    preds = pd.read_parquet(io_paths.preds_path("ext_finbert_llm", "rule", "5d"))
    longs = preds[preds.get("action", preds.get("traded")).astype(str).isin(["long", "1"])]
    by_tic = longs.assign(hit=(longs["excess_5d"] > 0).astype(int)).groupby("ticker").agg(
        n_long=("hit", "count"),
        hit_rate=("hit", "mean"),
        avg_excess=("excess_5d", "mean"),
    )
    print("Per-ticker rule hit rate, ext_finbert_llm +5d (test set only):")
    print(by_tic.to_string())
else:
    print("skip (ext_finbert_llm +5d not in this phase's grid)")

Per-ticker rule hit rate, ext_finbert_llm +5d (test set only):
        n_long  hit_rate  avg_excess
ticker                              
AMD          2  1.000000    0.092169
AVGO         3  0.666667    0.012741
BLK          4  0.500000   -0.015935
C            2  1.000000    0.040613
FAST         2  0.500000   -0.002202
FDX          4  0.750000    0.017175
GS           3  0.333333   -0.004368
JNJ          3  1.000000    0.022817
JPM          3  0.666667    0.002053
NKE          2  0.500000   -0.011448
NVDA         1  0.000000   -0.070398
PLTR         2  0.500000    0.003226
WFC          3  0.333333   -0.002550


## Stage A1 — Per-ticker cross-quarter narrative (PDF §6 deliverable)

PDF §6 asks "for 2-3 companies, a paragraph showing the story your pipeline tells across quarters". We pick **NVDA**, **INTC**, **FDX** — three different sectors with three different cross-quarter dynamics in this corpus. The narrative payload (sentiment trajectory, top wins / risks per quarter, guidance moves, qa-only risks) is persisted to `data/cache/analysis/narratives.json`; the report writer composes prose from it.

In [10]:
NARRATIVE_TICKERS = ["NVDA", "INTC", "FDX"]
narratives_path = analysis.write_narratives(NARRATIVE_TICKERS)
print(f"narratives -> {narratives_path}")
import json
nar = json.loads(narratives_path.read_text())
for tic in NARRATIVE_TICKERS:
    s = nar[tic]["summary"]
    n = nar[tic]["quarters"]
    print(f"\n[{tic}] {len(n)} calls   avg_sent={s['avg_sentiment']:+.3f}   "
          f"raised={s['n_guidance_raised']}  maintained={s['n_guidance_maintained']}  "
          f"lowered={s['n_guidance_lowered']}  qa_only_risks_total={s['qa_only_risks_total']}")
    for q in n:
        wins = "; ".join(q["top_wins"][:2]) if q["top_wins"] else "—"
        risks = "; ".join(q["top_risks"][:2]) if q["top_risks"] else "—"
        print(f"  {q['quarter']:8s} {q['call_date']}  sent={q['sentiment_call']:+.3f}  "
              f"g={q['guidance_call']:11s}  wins=[{wins[:60]}]  risks=[{risks[:60]}]")

narratives -> /home/tian/code/baruch/NLP_HW/HW1/data/cache/analysis/narratives.json

[NVDA] 9 calls   avg_sent=+0.440   raised=1  maintained=3  lowered=1  qa_only_risks_total=64
  Q4-2024  2024-02-21  sent=+0.410  g=lowered      wins=[conference call; earnings release]  risks=[forward-looking statements; significant risks and uncertaint]
  Q1-2025  2024-05-22  sent=+0.376  g=none         wins=[keynote; presentation]  risks=[forward-looking statements; uncertainties]
  Q2-2025  2024-08-28  sent=+0.426  g=maintained   wins=[Investor Relations call; Goldman Sachs Conference]  risks=[Significant risks and uncertainties; Materially differ]
  Q3-2025  2024-11-20  sent=+0.431  g=maintained   wins=[conference call; earnings release]  risks=[materially differ; supply constraints]
  Q4-2025  2025-02-26  sent=+0.400  g=maintained   wins=[conference call; earnings release]  risks=[significant risks and uncertainties; tariff impact]
  Q1-2026  2025-05-28  sent=+0.394  g=none         wins=[welcome c

## Stage A2 — Reactive vs proactive (PDF §6 extra credit)

Per PDF §4 Task 2 stretch goal: "a topic that an analyst raises without management having mentioned it in prepared remarks is a red flag (this is a real S&P research result)." We already capture `qa_minus_pres_risks` in every call JSON (`dispersion.qa_minus_pres_risks`); this stage aggregates them into a corpus-level table + summary so the report can cite the most "reactive" calls explicitly.

In [11]:
rp_df = analysis.reactive_proactive_table()
rp_summary = analysis.reactive_proactive_summary(rp_df)
print(f"corpus avg presenter risks per call: {rp_summary['corpus_avg_pres_risks']:.2f}")
print(f"corpus avg QA-only risks per call:   {rp_summary['corpus_avg_qa_only']:.2f}")
print("\nTop-10 most 'reactive' calls (analysts raised many risks NOT in prepared remarks):")
for r in rp_summary["top_reactive_calls"]:
    print(f"  {r['ticker']:5s} {r['quarter']:8s} {r['call_date']}  "
          f"qa_only={r['n_qa_only_risks']:2d}  e.g. [{r['top_qa_only_risks'][:80]}]")
print("\nPer-ticker average QA-only risks per call:")
for r in sorted(rp_summary["by_ticker_avg"], key=lambda x: -x["avg_qa_only"]):
    print(f"  {r['ticker']:5s}  n_calls={int(r['n_calls']):2d}  "
          f"avg_qa_only={r['avg_qa_only']:.2f}  avg_pres={r['avg_pres']:.2f}")

corpus avg presenter risks per call: 5.00
corpus avg QA-only risks per call:   10.18

Top-10 most 'reactive' calls (analysts raised many risks NOT in prepared remarks):
  FDX   Q1-2025  2024-09-19  qa_only=18  e.g. [asian shipper volume shifts; challenging u s domestic industrial economy impacti]
  FDX   Q2-2025  2024-12-19  qa_only=18  e.g. [base rate pressure; customer attrition; customer attrition as a result of separa]
  FDX   Q3-2026  2026-03-19  qa_only=18  e.g. [anomaly in the market; decelerating shipment rate of change; fuel price; fuel su]
  INTC  Q1-2025  2025-04-24  qa_only=18  e.g. [7nm constraint; arm competition in head nodes; competition from arm in head node]
  INTC  Q2-2025  2025-07-24  qa_only=18  e.g. [14a delay; 14a hedging; competition from arm taking over half the server market;]
  FDX   Q3-2025  2025-03-20  qa_only=17  e.g. [b2b weakness; b2b weakness in industrial production; challenges in europe not ye]
  INTC  Q4-2023  2024-01-25  qa_only=17  e.g. [backside p

## Stage A3 — gemma3:4b vs llama3.1:8b head-to-head (PDF §6 extra credit)

PDF §6 explicitly lists "Comparison of two different LLMs (e.g. Gemma 3 4B vs. Qwen 3 14B) on the same extraction task." Both LLMs ran on every call (Stage C). `consensus.guidance_by_model` carries each model's call-level guidance label; we compute per-call agreement and roll up per ticker for the figure.

In [12]:
llm_call_df = analysis.llm_agreement_per_call()
llm_tic_df = analysis.llm_agreement_per_ticker(llm_call_df)
overall_agree = llm_call_df["agree"].mean()
print(f"Corpus-wide gemma3:4b vs llama3.1:8b guidance-label agreement: "
      f"{int(llm_call_df['agree'].sum())}/{len(llm_call_df)}  "
      f"({overall_agree:.1%})")
print("\nGuidance label distribution per model (corpus):")
import collections
gemma_dist = collections.Counter(llm_call_df["gemma3_4b"].dropna())
llama_dist = collections.Counter(llm_call_df["llama3_1_8b"].dropna())
for k in ("raised", "maintained", "lowered", "none"):
    print(f"  {k:12s}  gemma={gemma_dist.get(k, 0):3d}  llama={llama_dist.get(k, 0):3d}")
print("\nPer-ticker agreement rate (sorted ascending — disagreement-prone tickers first):")
for _, r in llm_tic_df.sort_values("agree_rate").iterrows():
    print(f"  {r['ticker']:5s}  n_calls={int(r['n_calls']):2d}  "
          f"agree={int(r['n_agree']):2d}/{int(r['n_calls']):2d}  rate={r['agree_rate']:.2f}")

Corpus-wide gemma3:4b vs llama3.1:8b guidance-label agreement: 60/131  (45.8%)

Guidance label distribution per model (corpus):
  raised        gemma= 71  llama= 23
  maintained    gemma= 12  llama= 15
  lowered       gemma= 20  llama= 12
  none          gemma= 28  llama= 81

Per-ticker agreement rate (sorted ascending — disagreement-prone tickers first):
  BLK    n_calls=10  agree= 1/10  rate=0.10
  GS     n_calls=10  agree= 1/10  rate=0.10
  C      n_calls= 9  agree= 2/ 9  rate=0.22
  FDX    n_calls= 9  agree= 3/ 9  rate=0.33
  WFC    n_calls= 9  agree= 3/ 9  rate=0.33
  INTC   n_calls= 9  agree= 4/ 9  rate=0.44
  NKE    n_calls= 9  agree= 4/ 9  rate=0.44
  AMD    n_calls= 9  agree= 5/ 9  rate=0.56
  PLTR   n_calls= 9  agree= 5/ 9  rate=0.56
  JNJ    n_calls=10  agree= 6/10  rate=0.60
  NVDA   n_calls= 9  agree= 6/ 9  rate=0.67
  AVGO   n_calls= 9  agree= 6/ 9  rate=0.67
  FAST   n_calls=10  agree= 7/10  rate=0.70
  JPM    n_calls=10  agree= 7/10  rate=0.70


## Stage A4 — Cross-sectional long-short (PDF §6 extra credit)

PDF §6 extra credit: "Cross-sectional long-short portfolio: rank the 14 names each quarter by your signal, long the top few vs. short the bottom few." Each calendar quarter we rank every available ticker by `score = sentiment_call + 0.5 · guidance_ordinal` and PnL = mean(top-3 excess +5d) − mean(bottom-3 excess +5d). This is sector-neutral and tape-neutral (uses excess vs SPY) so the long-short cancels market beta at the basket level too.

In [13]:
cs_df, cs_summary = analysis.cross_sectional_backtest(feat_df, labels_df, horizon="5d", top_k=3)
print(f"Cross-sectional L/S top-3/bottom-3 on +5d excess  "
      f"({cs_summary['n_quarters']} quarters of corpus history)")
print(f"  mean per-quarter L/S return: {cs_summary['mean_ret_ls']:+.4f}")
print(f"  hit rate (quarters profitable): {cs_summary['hit_rate_quarters']:.2f}")
sharpe_str = f"{cs_summary['sharpe_quarterly']:.2f}" if cs_summary['sharpe_quarterly'] is not None else "n/a"
print(f"  quarterly Sharpe (sqrt(N) scaled): {sharpe_str}")
print(f"  final equity: {cs_summary['final_equity']:.3f}")
print("\nPer-quarter detail:")
print(cs_df[["quarter", "n_universe", "ret_long_avg", "ret_short_avg", "ret_ls",
             "equity", "long_tickers", "short_tickers"]].to_string(index=False))

Cross-sectional L/S top-3/bottom-3 on +5d excess  (9 quarters of corpus history)
  mean per-quarter L/S return: -0.0125
  hit rate (quarters profitable): 0.11
  quarterly Sharpe (sqrt(N) scaled): -1.80
  final equity: 0.891

Per-quarter detail:
quarter  n_universe  ret_long_avg  ret_short_avg    ret_ls   equity long_tickers short_tickers
 2024Q1          14      0.019746       0.027231 -0.007485 0.992515  GS,PLTR,FDX     JNJ,C,WFC
 2024Q2          14      0.008904       0.025264 -0.016360 0.976277 AMD,BLK,PLTR   NKE,JPM,WFC
 2024Q3          13     -0.010849      -0.000287 -0.010562 0.965966   BLK,AMD,GS  INTC,FDX,WFC
 2024Q4          15     -0.016332      -0.000590 -0.015742 0.950760   BLK,AMD,GS     C,NKE,WFC
 2025Q1          14      0.002491       0.009799 -0.007308 0.943812   GS,BLK,AMD INTC,FDX,FAST
 2025Q2          14      0.063686       0.031505  0.032181 0.974184   AMD,BLK,GS FAST,WFC,INTC
 2025Q3          14     -0.000285       0.041353 -0.041638 0.933621   C,NVDA,BLK PLTR,AMD,

## Stage A5 — Momentum-only baseline (alone vs combined)

PDF §6 extra credit: "Integration of an external signal (price momentum, analyst ratings from yfinance) with your NLP signal and a comparison of each alone vs. combined." `ret_21d_prior` is already a feature inside the NLP-combined models (logreg / xgb / ridge); this cell isolates it as a standalone signal so we can compare to those numbers in the report.

In [14]:
mom_5d = analysis.momentum_only_baseline(feat_df, labels_df, horizon="5d")
mom_21d = analysis.momentum_only_baseline(feat_df, labels_df, horizon="21d")
print("Momentum-only baseline: long when ret_21d_prior > 0 (entire corpus, no train/test split).")
for tag, m in (("+5d ", mom_5d), ("+21d", mom_21d)):
    sh = f"{m['sharpe']:.2f}" if m['sharpe'] is not None else "n/a"
    hr = f"{m['hit_rate']:.3f}" if m['hit_rate'] is not None else "n/a"
    print(f"  {tag}  n_trades={m['n_trades']:3d}  pnl_sum={m['pnl_sum']:+.3f}  "
          f"hit_rate={hr}  sharpe={sh}  final_equity={m['final_equity']:.3f}")

Momentum-only baseline: long when ret_21d_prior > 0 (entire corpus, no train/test split).
  +5d   n_trades= 79  pnl_sum=+1.084  hit_rate=0.633  sharpe=2.49  final_equity=2.685
  +21d  n_trades= 79  pnl_sum=+1.737  hit_rate=0.595  sharpe=2.24  final_equity=4.230


## Stage A6 — Render figures + bundle analysis summary

PDF §6 deliverable: "at least one equity-curve plot". Renders 5 PNGs under `data/cache/plots/` and bundles every analysis output into `data/cache/analysis/analysis_summary.json` for the report writer.

| Figure                | What it shows                                                        |
|-----------------------|----------------------------------------------------------------------|
| `equity_curves.png`   | Out-of-sample excess equity at +5d and +21d for the headline cells   |
| `raw_vs_spy.png`      | Strategy raw return compounded vs SPY buy-and-hold                   |
| `ic_grid.png`         | Spearman IC heatmap across (extraction × model × horizon)            |
| `llm_agreement.png`   | Per-ticker gemma3:4b vs llama3.1:8b guidance-label agreement         |
| `cross_sectional.png` | Quarterly long-short on top-3 / bottom-3 by NLP score                |

In [15]:
rendered = plots.render_all(horizons=("5d", "21d"))
print("Rendered figures:")
for name, p in rendered.items():
    print(f"  {name:20s}  {p}")

summary_path = analysis.write_combined_summary(
    narratives_path=narratives_path,
    rp_summary=rp_summary,
    llm_per_ticker=llm_tic_df,
    cs_summary=cs_summary,
    momentum_summary={"5d": mom_5d, "21d": mom_21d},
)
print(f"\nbundled analysis summary -> {summary_path}")

Rendered figures:
  equity_curves         /home/tian/code/baruch/NLP_HW/HW1/data/cache/plots/equity_curves.png
  raw_vs_spy            /home/tian/code/baruch/NLP_HW/HW1/data/cache/plots/raw_vs_spy.png
  ic_grid               /home/tian/code/baruch/NLP_HW/HW1/data/cache/plots/ic_grid.png
  llm_agreement         /home/tian/code/baruch/NLP_HW/HW1/data/cache/plots/llm_agreement.png
  cross_sectional       /home/tian/code/baruch/NLP_HW/HW1/data/cache/plots/cross_sectional.png

bundled analysis summary -> /home/tian/code/baruch/NLP_HW/HW1/data/cache/analysis/analysis_summary.json
